# ATLAS — Colab measurements using the real GitHub repository

This notebook clones `zakir-si/ATLAS`, imports its actual `atlas.models` and `atlas.benchmark` modules, runs a small LoRA baseline, and exports a measurement manifest. The baseline is **not** an ATLAS acceleration algorithm. No quality score is invented: the benchmark comparison requires a task-specific score normalized to `[0, 1]`.

## Cell 1 — Select a GPU

In Colab choose **Runtime → Change runtime type → GPU**. A T4 is fine if available. Run the next cell to verify the accelerator.

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Select a GPU runtime in Colab and rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## Cell 2 — Clone ATLAS from GitHub

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_DIR = Path('/content/ATLAS')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/zakir-si/ATLAS.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
REVISION = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
print('ATLAS repository:', REPO_DIR)
print('ATLAS commit:', REVISION)

## Cell 3 — Install training dependencies

This leaves Colab's preinstalled PyTorch in place, removes the incompatible optional `torchao` package if present, and installs the training libraries.

In [ ]:
%pip -q uninstall -y torchao
%pip -q install 'transformers>=4.46,<5' 'datasets>=3,<5' 'peft>=0.13,<1' 'accelerate>=1,<2' 'safetensors>=0.4'

In [ ]:
import importlib.metadata as md
from atlas.models import RunResult
from atlas.benchmark import compare_repeated_runs

print('Imported ATLAS RunResult:', RunResult)
print('Imported ATLAS benchmark:', compare_repeated_runs)
for package in ['transformers', 'datasets', 'peft', 'accelerate']:
    print(f'{package}:', md.version(package))

## Cell 4 — Configure a small baseline

This is a short smoke-test configuration. Increase the training budget only after the full run works. Check model and dataset terms before use.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B'
DATASET_ID = 'Salesforce/wikitext'
DATASET_CONFIG = 'wikitext-2-raw-v1'
TEXT_FIELD = 'text'
SEED = 42
MAX_LENGTH = 128
MAX_STEPS = 20
TRAIN_EXAMPLES = 1000
EVAL_EXAMPLES = 100
RUN_NAME = 'atlas-baseline-seed42'
OUTPUT_DIR = '/content/atlas-output'
print('Run:', RUN_NAME, '| steps:', MAX_STEPS)

In [ ]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID, DATASET_CONFIG)
train_raw = raw['train'].select(range(min(TRAIN_EXAMPLES, len(raw['train']))))
eval_raw = raw['validation'].select(range(min(EVAL_EXAMPLES, len(raw['validation']))))
train_raw = train_raw.filter(lambda row: bool(str(row[TEXT_FIELD]).strip()))
eval_raw = eval_raw.filter(lambda row: bool(str(row[TEXT_FIELD]).strip()))
if not len(train_raw) or not len(eval_raw):
    raise ValueError('No usable train/evaluation text rows found.')
print('Train rows:', len(train_raw), 'Eval rows:', len(eval_raw))
print('Dataset fingerprint:', train_raw._fingerprint)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_batch(batch):
    return tokenizer([str(x) for x in batch[TEXT_FIELD]], truncation=True, max_length=MAX_LENGTH)

train_ds = train_raw.map(tokenize_batch, batched=True, remove_columns=train_raw.column_names)
eval_ds = eval_raw.map(tokenize_batch, batched=True, remove_columns=eval_raw.column_names)
model_kwargs = {'low_cpu_mem_usage': True}
if torch.cuda.is_available():
    model_kwargs['torch_dtype'] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
base_model.config.use_cache = False
lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16, lora_dropout=0.05, target_modules='all-linear')
model = get_peft_model(base_model, lora_config)
model.gradient_checkpointing_disable()
model.print_trainable_parameters()
if not any(p.requires_grad for p in model.parameters()):
    raise RuntimeError('No trainable parameters found.')
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
from transformers import Trainer, TrainingArguments
from pathlib import Path
import time

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
args = TrainingArguments(
    output_dir=OUTPUT_DIR, run_name=RUN_NAME, max_steps=MAX_STEPS,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, learning_rate=2e-4, warmup_ratio=0.05,
    logging_steps=5, eval_strategy='steps', eval_steps=10,
    save_strategy='no', gradient_checkpointing=False,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to='none', seed=SEED, data_seed=SEED,
    dataloader_num_workers=0, remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collator)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
train_start = time.perf_counter()
train_output = trainer.train()
if torch.cuda.is_available(): torch.cuda.synchronize()
training_seconds = time.perf_counter() - train_start
if torch.cuda.is_available(): torch.cuda.synchronize()
eval_start = time.perf_counter()
eval_metrics = trainer.evaluate()
if torch.cuda.is_available(): torch.cuda.synchronize()
evaluation_seconds = time.perf_counter() - eval_start
gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
peak_memory_gb = round(torch.cuda.max_memory_allocated() / 1e9, 4)
print('Training seconds:', round(training_seconds, 2))
print('Evaluation seconds:', round(evaluation_seconds, 2))
print('GPU:', gpu_name, '| peak allocated GB:', peak_memory_gb)
print('Eval metrics:', eval_metrics)

## Cell 5 — Export measurements and reference ATLAS's data model

The manifest records real run measurements and the exact ATLAS Git commit. A task-specific quality score is intentionally not fabricated from validation loss.

In [ ]:
import json, platform
from datetime import datetime, timezone
from pathlib import Path

manifest = {
    'schema_version': 1,
    'run_name': RUN_NAME,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'atlas_repository': 'https://github.com/zakir-si/ATLAS',
    'atlas_code_revision': REVISION,
    'model_id': MODEL_ID,
    'model_revision': getattr(model.config, '_commit_hash', None),
    'dataset_id': DATASET_ID,
    'dataset_config': DATASET_CONFIG,
    'dataset_fingerprint': train_raw._fingerprint,
    'seed': SEED,
    'max_steps': MAX_STEPS,
    'max_length': MAX_LENGTH,
    'train_examples': len(train_ds),
    'eval_examples': len(eval_ds),
    'lora_rank': 8,
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'transformers_version': md.version('transformers'),
    'datasets_version': md.version('datasets'),
    'peft_version': md.version('peft'),
    'accelerate_version': md.version('accelerate'),
    'gpu_name': gpu_name,
    'gpu_count': gpu_count,
    'training_seconds': round(training_seconds, 4),
    'evaluation_seconds': round(evaluation_seconds, 4),
    'training_accelerator_hours': round(training_seconds * gpu_count / 3600, 6),
    'peak_memory_gb': peak_memory_gb,
    'train_metrics': {k: float(v) for k, v in train_output.metrics.items() if isinstance(v, (int, float))},
    'evaluation_metrics': {k: float(v) for k, v in eval_metrics.items() if isinstance(v, (int, float))},
    'task_quality_score': None,
    'task_quality_metric_name': None,
    'notes': 'Baseline only. Validation loss is diagnostic, not a normalized task-quality score.'
}
manifest_path = Path('/content/atlas-run-manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print('Saved:', manifest_path)

In [ ]:
from google.colab import files
files.download('/content/atlas-run-manifest.json')

## What this notebook uses from ATLAS

- Clones the actual repository and records its commit SHA.
- Imports `RunResult` from `src/atlas/models.py` and `compare_repeated_runs` from `src/atlas/benchmark.py`.
- Exports a machine-readable measurement manifest.

It does **not** claim the selector or benchmark helper accelerates training. The current comparison API requires a normalized task-quality score in `[0, 1]`; do not fill that field with validation loss. We should define a task-specific evaluator and an explicit manifest-to-`RunResult` adapter before running comparisons.